Agora estamos importando os dados para a camada Silver

In [2]:
import duckdb # type: ignore

In [3]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

Agora vamos agrupar por ID e ordenar por data_ingestao e depois fazer um inner select, que é select aninhado

In [4]:
df = con.execute("""
                 SELECT * 
                 FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao  DESC) As row
                    FROM bronze_z0019
                    WHERE data_ingestao >= '2025-01-01'
                 ) WhERE row = 1
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10002,MARTELO,BT50,100,1500,z0019_1.csv,2025-04-27 17:08:17.027682,1
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2025-04-27 18:28:57.620361,1
2,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2025-04-27 17:08:17.027682,1
3,10004,SERRA,BT50,100,200,z0019_2.csv,2025-04-27 18:28:57.620361,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2025-04-27 18:28:57.620361,1


Agora vamos passar esse dataframe para outro dataframe e fazendo a limpeza dos dados, vamos excluir as colunos que não precisamos sem alterar o dataframe principal.
Também aproveitar para renomear as colunas.

In [5]:
df_final = df.drop(columns=['nome_arquivo','data_ingestao','row'])
df_final = df_final.rename(columns={'NATBR':'id'})
df_final = df_final.rename(columns={'MAKTX':'nm_produto'})
df_final = df_final.rename(columns={'WERKS':'id_categoria'})
df_final = df_final.rename(columns={'MAINS':'id_fornecedor'})
df_final = df_final.rename(columns={'LABST':'vl_preco'})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,1500
1,10005,MACHADO,BT50,100,100
2,10001,PARAFUSO,BT10,100,100
3,10004,SERRA,BT50,100,200
4,10003,PREGO,BT10,100,60


Agora vamos verificar como esta tipado a os nossos dados, então usaremos na "df_final.dtypes"

In [6]:
df_final.dtypes


id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_preco         object
dtype: object

Verificamos que esta tudo como objeto, sctring, precisamos alteras o tipo para podermos efetuar cálculos.
vamos modicicar o df_final e aplicar uma função e criar e passar para outro dataframe

In [14]:
df2 =df_final
df2 = df2.astype(
    {
            'id': int,
            'nm_produto': str,
            'id_categoria': str,
            'id_fornecedor': int,
            'vl_preco': float,    }
)

df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,1500.0
1,10005,MACHADO,BT50,100,100.0
2,10001,PARAFUSO,BT10,100,100.0
3,10004,SERRA,BT50,100,200.0
4,10003,PREGO,BT10,100,60.0


Agora vamos verificar como ficou os formatos

In [15]:
df2.dtypes

id                 int64
nm_produto        object
id_categoria      object
id_fornecedor      int64
vl_preco         float64
dtype: object

Agora vamor criar a tabela produtos

In [17]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT,
            nm_produto TEXT,
            id_categoria TEXT,
            id_fornecedor BIGINT,
            vl_preco FLOAT
            )
""")
          

VAMOS VER COMO FICOU

In [18]:
df2.head(10)


,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,1500.0
1,10005,MACHADO,BT50,100,100.0
2,10001,PARAFUSO,BT10,100,100.0
3,10004,SERRA,BT50,100,200.0
4,10003,PREGO,BT10,100,60.0


VAMOS FAZER A CONSULTA DA VARIAVEL df_resultado

In [19]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco


aparece a tabela sem nenhuma dados, agora temos que popular a tabela e a ingestão desses dados

In [20]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

AGORA A TABELA VAI ESTAR POPULADA COM A INGESTÃO DOS DADOS DO DF2

In [21]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,1500.0
1,10005,MACHADO,BT50,100,100.0
2,10001,PARAFUSO,BT10,100,100.0
3,10004,SERRA,BT50,100,200.0
4,10003,PREGO,BT10,100,60.0


agora tem que fechar a conexão

In [22]:
con.close()